# Actividad evaluada: pipeline 16S simplificado

Este notebook ejecuta una versión mínima del análisis 16S con QIIME2:

1. Importar lecturas paired-end
2. Remover primers con cutadapt
3. Denoising con DADA2
4. Diversidad alfa/beta
5. Taxonomía

Completa las variables de la sección 0 con el dataset asignado a tu grupo.

## 0. Configuración

Edita solamente esta celda si necesitas cambiar de dataset o ajustar parámetros.

In [1]:
import os
from pathlib import Path

# Make sure QIIME2 uses the R installation inside the active conda/micromamba environment.
# This avoids rpy2 errors such as: shared object 'methods.dylib' not found.
CONDA_PREFIX = os.environ.get("CONDA_PREFIX", "")
if CONDA_PREFIX:
    os.environ["PATH"] = os.path.join(CONDA_PREFIX, "bin") + ":" + os.environ.get("PATH", "")
    r_home = os.path.join(CONDA_PREFIX, "lib", "R")
    if os.path.exists(r_home):
        os.environ["R_HOME"] = r_home
else:
    print("Warning: CONDA_PREFIX is empty. Open Jupyter from the qiime2-amplicon-2025.4 environment.")

# -------------------------------------------------------------------
# EDITAR ESTAS VARIABLES
# -------------------------------------------------------------------
GROUP_DATASET = "dataset_grupo_A"  # cambiar por dataset_grupo_A ... dataset_grupo_K

# Si el notebook se ejecuta desde la raíz del repositorio, esto funciona sin cambios.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "actividad_evaluada").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATASET_DIR = PROJECT_DIR / "actividad_evaluada" / GROUP_DATASET / "16S"
RAW_READS_DIR = DATASET_DIR / "raw_reads"
METADATA = DATASET_DIR / "metadata.tsv"
RESULTS_DIR = DATASET_DIR / "results"

# Primers 515F/806R, región V4
FWD_PRIMER = "GTGCCAGCMGCCGCGGTAA"
REV_PRIMER = "GGACTACHVGGGTWTCTAAT"
FWD_ADAPTER = "ATTAGAWACCCBDGTAGTCC"  # reverse complement de 806R
REV_ADAPTER = "TTACCGCGGCKGCTGGCAC"   # reverse complement de 515F

# Parámetros livianos para computadores pequeños
QIIME_THREADS = 1
DADA2_N_READS_LEARN = 400
TRUNC_F = 0
TRUNC_R = 0

# Ejecutar taxonomía por defecto. Si el computador tiene poca RAM, cambiar a False.
RUN_TAXONOMY = True
CLASSIFIER = PROJECT_DIR / "modulo1_16S_DADA2_QIIME2" / "data" / "taxonomy_db" / "silva-138-99-nb-classifier.qza"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset        : {GROUP_DATASET}")
print(f"Raw reads      : {RAW_READS_DIR}  exists={RAW_READS_DIR.exists()}")
print(f"Metadata       : {METADATA}  exists={METADATA.exists()}")
print(f"Results        : {RESULTS_DIR}")
print(f"Run taxonomy   : {RUN_TAXONOMY}")
print(f"R_HOME         : {os.environ.get('R_HOME', 'not set')}")
print(f"CONDA_PREFIX   : {CONDA_PREFIX or 'not set'}")

Dataset        : dataset_grupo_A
Raw reads      : /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/raw_reads  exists=True
Metadata       : /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/metadata.tsv  exists=True
Results        : /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results
Run taxonomy   : True
R_HOME         : /Users/fmelis/micromamba/envs/qiime2-amplicon-2025.4/lib/R
CONDA_PREFIX   : /Users/fmelis/micromamba/envs/qiime2-amplicon-2025.4


## 1. Crear manifest e importar lecturas

QIIME2 necesita un archivo `manifest.tsv` que indique dónde están los archivos FASTQ.

In [2]:
import glob
import pandas as pd

r1_files = sorted(glob.glob(str(RAW_READS_DIR / "*_R1.fastq.gz")))
if not r1_files:
    raise FileNotFoundError(f"No se encontraron archivos *_R1.fastq.gz en {RAW_READS_DIR}")

rows = []
for r1 in r1_files:
    sample_id = Path(r1).name.replace("_R1.fastq.gz", "")
    r2 = r1.replace("_R1.fastq.gz", "_R2.fastq.gz")
    if not Path(r2).exists():
        raise FileNotFoundError(f"Falta archivo R2 para {sample_id}: {r2}")
    rows.append({
        "sample-id": sample_id,
        "forward-absolute-filepath": str(Path(r1).resolve()),
        "reverse-absolute-filepath": str(Path(r2).resolve()),
    })

manifest = pd.DataFrame(rows)
MANIFEST = DATASET_DIR / "manifest.tsv"
manifest.to_csv(MANIFEST, sep="	", index=False)
print(f"Manifest creado: {MANIFEST}")
manifest

Manifest creado: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/manifest.tsv


,sample-id,forward-absolute-filepath,reverse-absolute-filepath
0,sample1,/Users/fmelis/Documents/Colabs/clases_sistemas...,/Users/fmelis/Documents/Colabs/clases_sistemas...
1,sample2,/Users/fmelis/Documents/Colabs/clases_sistemas...,/Users/fmelis/Documents/Colabs/clases_sistemas...
2,sample3,/Users/fmelis/Documents/Colabs/clases_sistemas...,/Users/fmelis/Documents/Colabs/clases_sistemas...
3,sample4,/Users/fmelis/Documents/Colabs/clases_sistemas...,/Users/fmelis/Documents/Colabs/clases_sistemas...


In [3]:
!qiime tools import \
    --type 'SampleData[PairedEndSequencesWithQuality]' \
    --input-path {MANIFEST} \
    --output-path {RESULTS_DIR}/sequences.qza \
    --input-format PairedEndFastqManifestPhred33V2

print("Secuencias importadas")

Imported /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/manifest.tsv as PairedEndFastqManifestPhred33V2 to /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/sequences.qza
Secuencias importadas


In [4]:
!qiime demux summarize \
    --i-data {RESULTS_DIR}/sequences.qza \
    --o-visualization {RESULTS_DIR}/sequences_summary.qzv

print("Resumen de calidad generado:", RESULTS_DIR / "sequences_summary.qzv")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/sequences_summary.qzv
Resumen de calidad generado: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/sequences_summary.qzv


## 2. Remover primers con cutadapt

In [5]:
!qiime cutadapt trim-paired \
    --i-demultiplexed-sequences {RESULTS_DIR}/sequences.qza \
    --p-front-f {FWD_PRIMER} \
    --p-front-r {REV_PRIMER} \
    --p-adapter-f {FWD_ADAPTER} \
    --p-adapter-r {REV_ADAPTER} \
    --p-times 2 \
    --p-discard-untrimmed \
    --p-cores {QIIME_THREADS} \
    --o-trimmed-sequences {RESULTS_DIR}/sequences_trimmed.qza

print("Primers removidos")

Saved SampleData[PairedEndSequencesWithQuality] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/sequences_trimmed.qza
Primers removidos


In [6]:
!qiime demux summarize \
    --i-data {RESULTS_DIR}/sequences_trimmed.qza \
    --o-visualization {RESULTS_DIR}/sequences_trimmed_summary.qzv

print("Resumen post-trimming generado")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/sequences_trimmed_summary.qzv
Resumen post-trimming generado


## 3. Denoising con DADA2

DADA2 filtra lecturas de baja calidad, corrige errores, une pares R1/R2 y remueve quimeras.

In [7]:
!qiime dada2 denoise-paired \
    --i-demultiplexed-seqs {RESULTS_DIR}/sequences_trimmed.qza \
    --p-trim-left-f 0 \
    --p-trim-left-r 0 \
    --p-trunc-len-f {TRUNC_F} \
    --p-trunc-len-r {TRUNC_R} \
    --p-n-threads {QIIME_THREADS} \
    --p-n-reads-learn {DADA2_N_READS_LEARN} \
    --o-table {RESULTS_DIR}/asv_table.qza \
    --o-representative-sequences {RESULTS_DIR}/rep_seqs.qza \
    --o-denoising-stats {RESULTS_DIR}/denoising_stats.qza

print("Denoising completado")

Saved FeatureTable[Frequency] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/asv_table.qza
Saved FeatureData[Sequence] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/rep_seqs.qza
Saved SampleData[DADA2Stats] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/denoising_stats.qza
Denoising completado


In [8]:
!qiime metadata tabulate \
    --m-input-file {RESULTS_DIR}/denoising_stats.qza \
    --o-visualization {RESULTS_DIR}/denoising_stats.qzv

!qiime feature-table summarize \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --m-sample-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/asv_table_summary.qzv

!qiime feature-table tabulate-seqs \
    --i-data {RESULTS_DIR}/rep_seqs.qza \
    --o-visualization {RESULTS_DIR}/rep_seqs.qzv

print("Tablas de DADA2 y ASVs generadas")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/denoising_stats.qzv
Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/asv_table_summary.qzv
Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/rep_seqs.qzv
Tablas de DADA2 y ASVs generadas


## 4. Diversidad

Para mantener este notebook liviano, usamos métricas no filogenéticas (`core-metrics`).

In [9]:
import zipfile

with zipfile.ZipFile(RESULTS_DIR / "denoising_stats.qza") as z:
    tsv = [f for f in z.namelist() if f.endswith(".tsv") and "stats" in f][0]
    lines = [line.decode() for line in z.open(tsv).readlines()]

# QIIME2/DADA2 stats files can vary slightly by version.
# Find the header line and then choose the best column for retained reads.
header_line_idx = next(i for i, line in enumerate(lines) if line.startswith("sample-id	"))
header = lines[header_line_idx].strip().split("	")
print("Columnas disponibles:", header)

preferred_columns = [
    "non-chimeric",
    "non-chimeric merged",
    "percentage of input non-chimeric",
    "merged",
    "denoised",
    "filtered",
]

idx = None
selected_column = None
for name in preferred_columns:
    if name in header:
        idx = header.index(name)
        selected_column = name
        break

if idx is None:
    numeric_candidates = [i for i, name in enumerate(header) if i > 0 and "percentage" not in name.lower()]
    if not numeric_candidates:
        raise ValueError(f"No se encontró una columna numérica usable en: {header}")
    idx = numeric_candidates[-1]
    selected_column = header[idx]

counts = {}
for line in lines[header_line_idx + 1:]:
    if not line.strip() or line.startswith("#"):
        continue
    parts = line.strip().split("	")
    if len(parts) <= idx:
        continue
    value = parts[idx].replace(",", "")
    try:
        counts[parts[0]] = int(float(value))
    except ValueError:
        continue

if not counts:
    raise ValueError("No se pudieron leer conteos desde denoising_stats.qza")

print(f"Usando columna para profundidad: {selected_column}")
print("Lecturas retenidas por muestra:")
for sample, count in counts.items():
    print(f"  {sample}: {count}")

positive_counts = {sample: count for sample, count in counts.items() if count > 0}
if not positive_counts:
    raise ValueError("Todas las muestras tienen 0 lecturas retenidas. Revisa trimming/DADA2.")

SAMPLING_DEPTH = min(positive_counts.values())
print()
print(f"SAMPLING_DEPTH = {SAMPLING_DEPTH}")

Columnas disponibles: ['sample-id', 'input', 'filtered', 'percentage of input passed filter', 'denoised', 'merged', 'percentage of input merged', 'non-chimeric', 'percentage of input non-chimeric']
Usando columna para profundidad: non-chimeric
Lecturas retenidas por muestra:
  sample1: 5
  sample2: 7
  sample3: 12
  sample4: 15

SAMPLING_DEPTH = 5


In [10]:
import shutil

diversity_dir = RESULTS_DIR / "diversity"
if diversity_dir.exists():
    shutil.rmtree(diversity_dir)

!qiime diversity core-metrics \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --p-sampling-depth {SAMPLING_DEPTH} \
    --m-metadata-file {METADATA} \
    --output-dir {diversity_dir}

print("Métricas de diversidad generadas")

Saved FeatureTable[Frequency] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/diversity/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/diversity/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/diversity/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/diversity/evenness_vector.qza
Saved DistanceMatrix to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/r

In [11]:
alpha_rarefaction = RESULTS_DIR / "alpha_rarefaction.qzv"
if alpha_rarefaction.exists():
    alpha_rarefaction.unlink()

!qiime diversity alpha-rarefaction \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --p-max-depth {SAMPLING_DEPTH} \
    --p-steps 4 \
    --m-metadata-file {METADATA} \
    --o-visualization {alpha_rarefaction}

print("Curvas de rarefacción generadas")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/alpha_rarefaction.qzv
Curvas de rarefacción generadas


In [12]:
import subprocess

try:
    subprocess.run([
        "qiime", "diversity", "alpha-group-significance",
        "--i-alpha-diversity", str(RESULTS_DIR / "diversity" / "shannon_vector.qza"),
        "--m-metadata-file", str(METADATA),
        "--o-visualization", str(RESULTS_DIR / "diversity" / "shannon_significance.qzv"),
    ], check=True)
    print("Test de diversidad alfa generado")
except subprocess.CalledProcessError:
    print("No se pudo generar el test de diversidad alfa. Revisa si quedaron suficientes muestras por grupo.")

try:
    subprocess.run([
        "qiime", "diversity", "beta-group-significance",
        "--i-distance-matrix", str(RESULTS_DIR / "diversity" / "bray_curtis_distance_matrix.qza"),
        "--m-metadata-file", str(METADATA),
        "--m-metadata-column", "Treatment",
        "--o-visualization", str(RESULTS_DIR / "diversity" / "bray_curtis_significance.qzv"),
    ], check=True)
    print("PERMANOVA Bray-Curtis generado")
except subprocess.CalledProcessError:
    print("No se pudo generar PERMANOVA. Revisa si quedaron suficientes muestras por grupo.")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/diversity/shannon_significance.qzv
Test de diversidad alfa generado
Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/diversity/bray_curtis_significance.qzv
PERMANOVA Bray-Curtis generado


## 5. Taxonomía

Esta sección usa `RUN_TAXONOMY = True` y requiere que el clasificador Silva esté disponible.

In [13]:
if RUN_TAXONOMY:
    if not CLASSIFIER.exists():
        raise FileNotFoundError(f"No se encontró el clasificador: {CLASSIFIER}")

    !qiime feature-classifier classify-sklearn \
        --i-classifier {CLASSIFIER} \
        --i-reads {RESULTS_DIR}/rep_seqs.qza \
        --p-n-jobs {QIIME_THREADS} \
        --o-classification {RESULTS_DIR}/taxonomy.qza

    !qiime metadata tabulate \
        --m-input-file {RESULTS_DIR}/taxonomy.qza \
        --o-visualization {RESULTS_DIR}/taxonomy.qzv

    !qiime taxa barplot \
        --i-table {RESULTS_DIR}/asv_table.qza \
        --i-taxonomy {RESULTS_DIR}/taxonomy.qza \
        --m-metadata-file {METADATA} \
        --o-visualization {RESULTS_DIR}/taxa_barplot.qzv

    print("Taxonomía generada")
else:
    print("Taxonomía omitida porque RUN_TAXONOMY = False")

Saved FeatureData[Taxonomy] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/taxonomy.qza
Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/taxonomy.qzv
Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/actividad_evaluada/dataset_grupo_A/16S/results/taxa_barplot.qzv
Taxonomía generada


## 6. Archivos generados

Los archivos `.qzv` se visualizan en <https://view.qiime2.org>.

In [14]:
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RESULTS_DIR))

alpha_rarefaction.qzv
asv_table.qza
asv_table_summary.qzv
denoising_stats.qza
denoising_stats.qzv
diversity/bray_curtis_distance_matrix.qza
diversity/bray_curtis_emperor.qzv
diversity/bray_curtis_pcoa_results.qza
diversity/bray_curtis_significance.qzv
diversity/evenness_vector.qza
diversity/jaccard_distance_matrix.qza
diversity/jaccard_emperor.qzv
diversity/jaccard_pcoa_results.qza
diversity/observed_features_vector.qza
diversity/rarefied_table.qza
diversity/shannon_significance.qzv
diversity/shannon_vector.qza
rep_seqs.qza
rep_seqs.qzv
sequences.qza
sequences_summary.qzv
sequences_trimmed.qza
sequences_trimmed_summary.qzv
taxa_barplot.qzv
taxonomy.qza
taxonomy.qzv
